In [38]:
# =========================
# Set up project path
# =========================

import sys
from src.config import (
    CLEANED_DATA_PATH,
    RESULTS_DIR,
    COUNTRY_SUMMARY_PATH,
    COUNTRY_UPLIFT_PATH
)
from statsmodels.stats.proportion import proportions_ztest
import numpy as np

# If this notebook is inside the notebooks/ folder
PROJECT_ROOT = Path.cwd().parent

# Add project root to Python path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Current working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("Project root exists:", PROJECT_ROOT.exists())

Current working directory: d:\2.DAP391m_Project\notebooks
Project root: d:\2.DAP391m_Project
Project root exists: True


In [39]:
# =========================
# Import Cleaned Dataset
# =========================

import pandas as pd
from src.config import CLEANED_DATA_PATH

print("Cleaned data path:", CLEANED_DATA_PATH)
print("File exists:", CLEANED_DATA_PATH.exists())

if not CLEANED_DATA_PATH.exists():
    raise FileNotFoundError(f"Cleaned dataset not found at: {CLEANED_DATA_PATH}")

df = pd.read_csv(CLEANED_DATA_PATH)

print("Cleaned dataset imported successfully!")
print("Shape:", df.shape)

display(df.head())

Cleaned data path: D:\2.DAP391m_Project\data\processed\dataset_cleaned.csv
File exists: True
Cleaned dataset imported successfully!
Shape: (288541, 9)


,user_id,timestamp,group,landing_page,converted,country,time_raw,elapsed_minutes,time_bucket
0,851104,11:48.6,control,old_page,0,US,11:48.6,11.810000,0-15_min
1,804228,01:45.2,control,old_page,0,US,01:45.2,1.753333,0-15_min
2,661590,55:06.2,treatment,new_page,0,US,55:06.2,55.103333,45-60_min
3,853541,28:03.1,treatment,new_page,0,US,28:03.1,28.051667,15-30_min
4,864975,52:26.2,control,old_page,1,US,52:26.2,52.436667,45-60_min


In [3]:
# Separate users who viewed the old page
old_page_data = df[df["landing_page"] == "old_page"]

# Separate users who viewed the new page
new_page_data = df[df["landing_page"] == "new_page"]

# Calculate conversion rate for old_page
# Since converted is 0 or 1, the mean of converted is the conversion rate
old_page_conversion_rate = old_page_data["converted"].mean()

# Calculate conversion rate for new_page
new_page_conversion_rate = new_page_data["converted"].mean()

# Calculate relative uplift of new_page compared with old_page
# This tells us how much new_page increases or decreases conversion rate relative to old_page
original_uplift = (
    (new_page_conversion_rate - old_page_conversion_rate)
    / old_page_conversion_rate
) * 100

# Print the original conversion rates and uplift
print(f"Old page CR: {old_page_conversion_rate * 100:.2f}%")
print(f"New page CR: {new_page_conversion_rate * 100:.2f}%")
print(f"Original uplift: {original_uplift:.2f}%")

Old page CR: 12.03%
New page CR: 11.87%
Original uplift: -1.30%


## Check Country Distribution

Before comparing conversion rates by country, we first check how many users are available in each country.

This step is important because segment size affects interpretation. A country with many users, such as the US, can strongly influence the overall result. A country with fewer users, such as CA, may show larger fluctuations because the sample size is smaller.

In [4]:
# Count number of users in each country
country_counts = df["country"].value_counts().reset_index()

# Rename columns for readability
country_counts.columns = ["country", "users"]

# Display country distribution
country_counts

,country,users
0,US,202186
1,UK,71961
2,CA,14394


## Calculate Conversion Rate by Country and Landing Page

Next, we calculate conversion rate separately for each country and each landing page.

This allows us to compare:

- CA old_page vs CA new_page
- UK old_page vs UK new_page
- US old_page vs US new_page

The goal is to check whether the new page performs consistently across different countries or whether its effect varies by market.

In [5]:
# Group data by country and landing_page
# For each group, calculate:
# - number of users
# - number of conversions
# - conversion rate
country_summary = df.groupby(["country", "landing_page"]).agg(
    users=("user_id", "count"),
    conversions=("converted", "sum"),
    conversion_rate=("converted", "mean")
).reset_index()

# Convert conversion rate from decimal to percentage
country_summary["conversion_rate_percent"] = (
    country_summary["conversion_rate"] * 100
)

# Display country-level conversion summary
country_summary

,country,landing_page,users,conversions,conversion_rate,conversion_rate_percent
0,CA,new_page,7256,811,0.111770,11.176957
1,CA,old_page,7138,848,0.118801,11.880078
2,UK,new_page,35861,4341,0.121051,12.105072
3,UK,old_page,36100,4330,0.119945,11.994460
4,US,new_page,101198,11982,0.118402,11.840155
5,US,old_page,100988,12171,0.120519,12.051927


## Create Country-Level Uplift Table

The country summary table has two rows for each country: one for `old_page` and one for `new_page`.

To make comparison easier, we pivot the table so that each country appears in one row, with separate columns for old page conversion rate and new page conversion rate.

This table will be used to calculate:

- old_page conversion rate by country
- new_page conversion rate by country
- absolute lift by country
- relative uplift by country

In [8]:
# Pivot the table so each country has one row
country_uplift = country_summary.pivot(
    index="country",
    columns="landing_page",
    values="conversion_rate"
).reset_index()

# Display pivoted table
country_uplift

landing_page,country,new_page,old_page
0,CA,0.111770,0.118801
1,UK,0.121051,0.119945
2,US,0.118402,0.120519


## Convert Conversion Rates to Percent

The pivot table stores conversion rates as decimals.

For example:

- 0.1200 means 12.00%
- 0.1180 means 11.80%

To make the table easier to interpret, we convert conversion rates into percentage format.

In [9]:
# Convert old_page conversion rate to percentage
country_uplift["old_page_percent"] = (
    country_uplift["old_page"] * 100
)

# Convert new_page conversion rate to percentage
country_uplift["new_page_percent"] = (
    country_uplift["new_page"] * 100
)

# Display updated table
country_uplift

landing_page,country,new_page,old_page,old_page_percent,new_page_percent
0,CA,0.111770,0.118801,11.880078,11.176957
1,UK,0.121051,0.119945,11.994460,12.105072
2,US,0.118402,0.120519,12.051927,11.840155


## Calculate Absolute Lift by Country

Absolute lift measures the direct difference between the new page conversion rate and the old page conversion rate.

Formula:

Absolute Lift = CR_new - CR_old

In this project, absolute lift is expressed in percentage points.

If the value is positive, the new page has a higher conversion rate.
If the value is negative, the old page has a higher conversion rate.

In [ ]:
# Calculate absolute lift in percentage points
country_uplift["absolute_lift_pp"] = (
    country_uplift["new_page_percent"]
    - country_uplift["old_page_percent"]
)

# Display updated table
country_uplift

landing_page,country,new_page,old_page,old_page_percent,new_page_percent,relative_uplift_percent,absolute_lift_pp
0,CA,0.111770,0.118801,11.880078,11.176957,-5.918492,-0.703121
1,UK,0.121051,0.119945,11.994460,12.105072,0.922197,0.110613
2,US,0.118402,0.120519,12.051927,11.840155,-1.757163,-0.211772


## Calculate Relative Uplift by Country

Relative uplift measures the percentage change of the new page compared with the old page.

Formula:

Relative Uplift = ((CR_new - CR_old) / CR_old) × 100

This metric answers:

"Compared with the old page, how much better or worse is the new page in each country?"

In [29]:
# Calculate relative uplift in percent
country_uplift["relative_uplift_percent"] = (
    (country_uplift["new_page"] - country_uplift["old_page"])
    / country_uplift["old_page"]
) * 100

# Display updated table
country_uplift

landing_page,country,new_page,old_page,old_page_percent,new_page_percent,relative_uplift_percent,absolute_lift_pp
0,CA,0.111770,0.118801,11.880078,11.176957,-5.918492,-0.703121
1,UK,0.121051,0.119945,11.994460,12.105072,0.922197,0.110613
2,US,0.118402,0.120519,12.051927,11.840155,-1.757163,-0.211772


## Identify the Better Page by Country

After calculating conversion rates and uplift, we identify which page performs better in each country.

This is a descriptive comparison:

- If new_page conversion rate is higher than old_page, then new_page is marked as better.
- If old_page conversion rate is higher than new_page, then old_page is marked as better.

This step does not yet prove statistical significance. It only summarizes the observed pattern.

In [30]:
# Identify which page has higher conversion rate in each country
country_uplift["better_page"] = country_uplift.apply(
    lambda row: "new_page" if row["new_page"] > row["old_page"] else "old_page",
    axis=1
)

# Display updated table
country_uplift

landing_page,country,new_page,old_page,old_page_percent,new_page_percent,relative_uplift_percent,absolute_lift_pp,better_page
0,CA,0.111770,0.118801,11.880078,11.176957,-5.918492,-0.703121,old_page
1,UK,0.121051,0.119945,11.994460,12.105072,0.922197,0.110613,new_page
2,US,0.118402,0.120519,12.051927,11.840155,-1.757163,-0.211772,old_page


## Sort Country Results by Relative Uplift

To make the result easier to interpret, we sort countries by relative uplift.

Countries at the top have the highest relative uplift for the new page.
Countries at the bottom have the lowest relative uplift for the new page.

This helps identify where the new page performs relatively better or worse.

In [31]:
# Sort countries by relative uplift from highest to lowest
country_uplift_sorted = country_uplift.sort_values(
    by="relative_uplift_percent",
    ascending=False
)

# Display sorted table
country_uplift_sorted

landing_page,country,new_page,old_page,old_page_percent,new_page_percent,relative_uplift_percent,absolute_lift_pp,better_page
1,UK,0.121051,0.119945,11.994460,12.105072,0.922197,0.110613,new_page
2,US,0.118402,0.120519,12.051927,11.840155,-1.757163,-0.211772,old_page
0,CA,0.111770,0.118801,11.880078,11.176957,-5.918492,-0.703121,old_page


## Save Country Results

Two result files are saved:

1. `country_summary.csv`  
   This file contains conversion rate by country and landing page.

2. `country_uplift.csv`  
   This file contains old page CR, new page CR, absolute lift, relative uplift, and the better page for each country.

These files will be used later for visualization, country-level hypothesis testing, and reporting.

In [40]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

country_summary.to_csv(
    COUNTRY_SUMMARY_PATH,
    index=False
)

country_uplift.to_csv(
    COUNTRY_UPLIFT_PATH,
    index=False
)

print(COUNTRY_SUMMARY_PATH.exists())
print(COUNTRY_UPLIFT_PATH.exists())

True
True


## Interpretation

Country-level segmentation was conducted to examine whether the effect of the new page varies across markets.

The results show that the new page does not perform consistently across countries.

- In the UK, the new page shows a small positive uplift.
- In the US, the new page shows a negative uplift.
- In CA, the new page shows the largest negative relative uplift.

This suggests that the treatment effect may vary by country. However, this is still a descriptive segmentation result. Statistical testing by country is required before making strong conclusions about country-specific effects.

## Summary

In this notebook, country-level segmentation was performed to analyze whether the new page behaves differently across countries.

The analysis followed these steps:

1. Check user distribution by country.
2. Calculate conversion rate by country and landing page.
3. Create a country-level uplift table.
4. Calculate absolute lift by country.
5. Calculate relative uplift by country.
6. Identify the better-performing page in each country.
7. Save country-level result files.

The output files from this notebook are:

- `country_summary.csv`
- `country_uplift.csv`

These results will be used in the next step for country-level hypothesis testing.